# 作业 03：在 PyTorch 中调用交叉熵

本次只判断一件事：**有没有行人？** 不使用图片、CNN 或数据下载。我们用一个简单函数产生两个预测概率，把注意力放在最后的损失函数调用上。

类别的顺序始终固定：

| 类别编号 | 含义 | one-hot 标签 y |
|---|---|---|
| 0 | 没有行人 | `[1, 0]` |
| 1 | 有行人 | `[0, 1]` |

`y` 和 `y_pred` 都表示**同一个样本的两个类别**，shape 为 `(2,)`。`y_pred[0]` 是没有行人的概率，`y_pred[1]` 是有行人的概率，两者相加为 1。

目标：会调用 `F.cross_entropy`，并能解释概率变大或变小时，损失为什么变化。


In [1]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

def show(name, value):
    print(f'{name}: {value}, shape={tuple(value.shape)}, dtype={value.dtype}')


## 1. 一个输出两个概率的预测函数

为了便于观察，直接把输入 x 当作“有行人的预测概率”，于是“没有行人的概率”就是 `1-x`。这是人为设置的教学函数，不是训练好的行人识别器。

先取 x=0.2，则预测应为 `[0.8, 0.2]`。

**本次实验 1：** 按文末要求，将下面代码中的 `x` 改为 `0.8`，并从本节运行到手算核对。原题的 `x=0.2` 是比较基准：预测为 `[0.8, 0.2]`、损失约为 `0.2231`；本次实验将观察修改后的结果。


In [2]:
def predict(x):
    # x 是一个 0 到 1 之间的浮点标量张量
    return torch.stack([1 - x, x])

x = torch.tensor(0.8)  # 实验 1：把有行人的预测概率改为 0.8
y = torch.tensor([1., 0.])  # 真实情况：没有行人
y_pred = predict(x)

show('x', x)
show('y（one-hot 真实标签）', y)
show('y_pred（两个预测概率）', y_pred)
show('预测概率之和', y_pred.sum())


x: 0.800000011920929, shape=(), dtype=torch.float32
y（one-hot 真实标签）: tensor([1., 0.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.2000, 0.8000]), shape=(2,), dtype=torch.float32
预测概率之和: 1.0, shape=(), dtype=torch.float32


## 2. 最后调用 cross_entropy 计算损失

**PyTorch 的 `F.cross_entropy` 接收原始分数 logits，而不是直接接收概率。**

本例的函数已经输出概率，因此先用 `torch.log(y_pred)` 构造一组等价分数：对这组分数做 softmax，会还原原来的概率。这里两个概率均大于 0。

真实标签使用类别编号：`y=[1,0]` 对应编号 `0`，`y=[0,1]` 对应编号 `1`。用 `y.argmax()` 取出这个编号，得到整数标量。这里取的是**真实标签**的编号，不是预测的 argmax。

请重点看最后一行：损失由 `F.cross_entropy(...)` 计算。


In [3]:
scores = torch.log(y_pred)  # 从本例的概率构造等价 logits
label = y.argmax()        # [1, 0] → 类别编号 0，dtype 为 torch.int64

show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))
show('label（真实类别编号）', label)

loss = F.cross_entropy(scores, label)
show('loss', loss)


scores（传给损失函数的分数）: tensor([-1.6094, -0.2231]), shape=(2,), dtype=torch.float32
softmax(scores)（还原概率）: tensor([0.2000, 0.8000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 0, shape=(), dtype=torch.int64
loss: 1.6094379425048828, shape=(), dtype=torch.float32


### 对照手算

展开公式：

$$L=-[y_0\ln(p_0)+y_1\ln(p_1)].$$

真实标签为 `[1,0]` 时，只留下 $-\ln(p_0)$。当前 $p_0=0.8$，所以损失约为 **0.2231**。

下面仅用于核对；实际调用损失函数时，仍使用上面的 `F.cross_entropy`。

**本次实验 1 的手算：** 上面的 `0.2231` 对应原题 `x=0.2`。修改为 `x=0.8` 后，真实类别仍为 0，其预测概率变为 `p0=0.2`，因此本次运行应得到 $-\ln(0.2)\approx1.6094$。


In [4]:
manual_loss = -torch.log(y_pred[label])
show('手算：真实类别概率的负对数', manual_loss)
print('与 cross_entropy 一致：', torch.allclose(loss, manual_loss))


手算：真实类别概率的负对数: 1.6094379425048828, shape=(), dtype=torch.float32
与 cross_entropy 一致： True


## 3. 几个带数字的例子

下面固定真实标签为 `[1,0]`，即“没有行人”。只改变模型的预测概率。

| 真实标签 y | 预测概率 y_pred | 交叉熵 |
|---|---|---|
| `[1,0]` | `[0.9,0.1]` | $-\ln(0.9)\approx0.1054$ |
| `[1,0]` | `[0.5,0.5]` | $-\ln(0.5)\approx0.6931$ |
| `[1,0]` | `[0.1,0.9]` | $-\ln(0.1)\approx2.3026$ |
| `[1,0]` | `[0,1]` | $-\ln(0)\to+\infty$：把真实类别的概率设为 0 |
| `[1,0]` | `[1,0]` | $-\ln(1)=0$：完全相信真实类别 |

概率越接近真实标签，损失越小。概率精确为 0 的情况是边界示例；PyTorch 会在下面的错误预测例子中打印 `inf`，表示正无穷。


In [5]:
def compare(y, probabilities):
    label = y.argmax()
    # 仅为展示指定概率的交叉熵。log(0)=-inf，是这里刻意保留的边界情况。
    scores = torch.log(probabilities)
    loss = F.cross_entropy(scores, label)
    print(f'y={y.tolist()}, y_pred={probabilities.tolist()}')
    print(f'真实类别概率={probabilities[label].item():.4f}, loss={loss.item():.4f}')
    return loss

y = torch.tensor([1., 0.])
example_losses = []
for values in [[0.9, 0.1], [0.5, 0.5], [0.1, 0.9], [0., 1.], [1., 0.]]:
    example_losses.append(compare(y, torch.tensor(values)))
    print()


y=[1.0, 0.0], y_pred=[0.8999999761581421, 0.10000000149011612]
真实类别概率=0.9000, loss=0.1054

y=[1.0, 0.0], y_pred=[0.5, 0.5]
真实类别概率=0.5000, loss=0.6931

y=[1.0, 0.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.1000, loss=2.3026

y=[1.0, 0.0], y_pred=[0.0, 1.0]
真实类别概率=0.0000, loss=inf

y=[1.0, 0.0], y_pred=[1.0, 0.0]
真实类别概率=1.0000, loss=0.0000



## 4. 换成“有行人”试试看

现在 y 为 `[0,1]`，真实类别是 1，应当关注第二个概率。下面预测为 `[0.1,0.9]`，损失约为 **0.1054**。同一个预测在上节被惩罚，在这里得到较小损失，是因为真实答案变了。


In [6]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.1, 0.9])
compare(y, y_pred)


y=[0.0, 1.0], y_pred=[0.10000000149011612, 0.8999999761581421]
真实类别概率=0.9000, loss=0.1054


tensor(0.1054)

## 5. 真正的网络输出原始分数时，直接调用

前面的 `torch.log` 是为了配合本题“给定概率”的数值演示。真实网络通常直接输出两个原始分数，此时**直接把分数交给 cross_entropy**，不需要先 softmax 或取 log。

下面的 scores 为 `[2,-1]`，它们不是概率，也不需要加起来等于 1。softmax 只用来把概率打印给我们看，不是交叉熵的输入。


In [7]:
scores = torch.tensor([2., -1.])
y = torch.tensor([1., 0.])
label = y.argmax()

show('原始分数 scores', scores)
show('给人看的预测概率', scores.softmax(dim=0))
loss = F.cross_entropy(scores, label)
show('直接调用 cross_entropy 得到的 loss', loss)


原始分数 scores: tensor([ 2., -1.]), shape=(2,), dtype=torch.float32
给人看的预测概率: tensor([0.9526, 0.0474]), shape=(2,), dtype=torch.float32
直接调用 cross_entropy 得到的 loss: 0.04858732968568802, shape=(), dtype=torch.float32


## 你的实验（50分）

1. 把第 1 节的 x 改成 `0.8`，从第 1 节重新运行到手算核对。保持真实标签为 `[1,0]`。记录 y_pred 和损失，解释为什么损失变大。
2. 在下面补全代码：真实标签是“有行人”，预测概率为 `[0.8,0.2]`。打印两个张量的 shape，再调用 cross_entropy，预期损失约为 `1.6094`。
3. 用自己的话解释：为什么 `[1,0]` 对 `[0,1]` 的损失是无穷大，而 `[1,0]` 对 `[1,0]` 的损失是 0？
4. `F.cross_entropy(y_pred, label)` 与 `F.cross_entropy(torch.log(y_pred), label)` 中，哪一个才是在计算这里指定的预测概率对应的交叉熵？原始网络输出 scores 时，又该怎样调用？


In [8]:
y = torch.tensor([0., 1.])       # 真实情况：有行人
y_pred = torch.tensor([0.8, 0.2])  # 对有行人的预测概率只有 0.2

show('y（one-hot 真实标签）', y)
show('y_pred（两个预测概率）', y_pred)

label = y.argmax()            # 由真实标签得到类别 1，整数标量
scores = torch.log(y_pred)    # 给定概率转换为等价 logits
show('label（真实类别编号）', label)
show('scores（等价原始分数）', scores)

loss = F.cross_entropy(scores, label)
show('loss', loss)
manual_loss = -torch.log(y_pred[label])
show('手算核对', manual_loss)
print('与手算一致：', torch.allclose(loss, manual_loss))


y（one-hot 真实标签）: tensor([0., 1.]), shape=(2,), dtype=torch.float32
y_pred（两个预测概率）: tensor([0.8000, 0.2000]), shape=(2,), dtype=torch.float32
label（真实类别编号）: 1, shape=(), dtype=torch.int64
scores（等价原始分数）: tensor([-0.2231, -1.6094]), shape=(2,), dtype=torch.float32
loss: 1.6094379425048828, shape=(), dtype=torch.float32
手算核对: 1.6094379425048828, shape=(), dtype=torch.float32
与手算一致： True


### 我的回答 (50分)

- **修改 x 后的预测与损失：** 将第 1 节 `x` 从 `0.2` 改为 `0.8` 后，`predict(x)` 得到 `y_pred≈[0.2, 0.8]`，shape 为 `(2,)`。真实标签保持 `[1, 0]`，因此真实类别为 0，损失为 $-\ln(0.2)\approx1.6094$，与手算一致。原先真实类别的预测概率为 `0.8`，损失约为 `0.2231`；修改后真实类别的概率降到 `0.2`，模型更倾向于错误类别，所以损失变大。

- **“有行人”实验的结果：** `y=[0, 1]`、`y_pred=[0.8, 0.2]`，两个张量的 shape 都是 `(2,)`，表示同一个样本的两个类别。`label=y.argmax()` 得到整数标量 `1`，shape 为 `()`、dtype 为 `torch.int64`。`scores=log(y_pred)≈[-0.2231, -1.6094]`，调用 `F.cross_entropy(scores, label)` 后得到标量损失约为 `1.6094`，shape 为 `()`，因为有行人这一真实类别的预测概率只有 `0.2`。

- **两个极端例子的解释：** 真实标签 `[1, 0]` 表示类别 0。若预测为 `[0, 1]`，真实类别的预测概率为 `0`，损失 $-\ln(p_0)$ 在 $p_0\to0^+$ 时趋于正无穷，所以程序显示 `inf`，表示非常自信地预测错了。若预测为 `[1, 0]`，真实类别概率为 `1`，损失为 $-\ln(1)=0$。手算时只取真实类别的负对数，避免直接计算 `0 * log(0)` 产生 `nan`；本题代码先将真实标签转为类别编号来计算损失。

- **概率与原始分数应该怎样传入损失函数：** 本题已经指定了预测概率，因此应使用 `F.cross_entropy(torch.log(y_pred), label)`，它内部的 softmax 会将这些等价分数还原为原来的概率。直接用 `F.cross_entropy(y_pred, label)` 会把概率值当成原始分数，再做一次 softmax，计算的就不是题目给定概率对应的交叉熵。例如真实类别为 1、给定概率为 `[0.8, 0.2]` 时，正确损失约为 `1.6094`，直接把概率当分数传入则约为 `1.0375`。真实网络如果已经输出原始分数 `scores`，应直接调用 `F.cross_entropy(scores, label)`，不需要先 softmax 或取 log；标签编号来自真实标签，不能用预测类别代替。

### 提交

补全实验代码和文字回答，重启内核后运行全部单元格并保存输出。示例中的 `inf` 是刻意展示的无穷大，不是代码异常。本题不要求编写训练循环。

参考：[PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)。
